In [ ]:
include( "../model.jl" )
include( "../cluster.jl" )
include( "../prob.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/nos-list.jl" )

using Optim
using ThreadsX

home, base = expfolder( "system" )

colormap = cgrad( :ice; rev=true )

In [ ]:
# Hyper parameters.
N = 128;  s = 1.0

# Create folder and extract critical noise.
folderlist = [createfolder( findfolder( N, ρ, s; base=base, warn=false, crit=true ) ) for ρ ∈ ρlist]

# Extract cluster mass distribution exponent.
λlist = vec( first.( readdlm.( folderlist.*"power-law.txt" ) ) )

# Import the cluster change frequency values.
θdata = readdlm.( folderlist.*"optimal-rates.txt" )

# Extract the indeces with data points.
# Note: Also enforcing upper bound.
ιdata = [BitVector( [
    all( [isfinite( θdata[i][j,k] ) & (θdata[i][j,k] < 10.0) &
        (k < 3 ? 0.1 < θdata[i][j,k] : true) for k ∈ 1:3] )
    for j ∈ 1:N] ) for i ∈ 1:Nρ];

In [ ]:
# Left and right distributions expectation.
λ1list = [mean( θdata[i][ιdata[i],1] ) for i ∈ 1:Nρ]
λ2list = [mean( θdata[i][ιdata[i],2] ) for i ∈ 1:Nρ]
κlist  = [mean( θdata[i][ιdata[i],3] ) for i ∈ 1:Nρ];

In [ ]:
i = 1

# Plot the optimal rates for changing cluster size.
plt = plot(; layout=(3,1), size=(400,600), legend=false )

for j ∈ 1:2
    # Exponent for changing cluster size.
    scatter!( plt[j], 1:N, θdata[i][:,j]; color=:gray )

    # Exponent from the cluster mass distribution.
    plot!( plt[j], [1,N], fill( λlist[i], 2 ); color=:indianred, linestyle=:dash, lw=2 )

    # Plot details.
    plot!( plt[j]; xlims=(1,N), ylims=(0,3) )
end

# Add exected value.
plot!( plt[1], [1,N], fill( λ1list[i], 2 ); color=:cornflowerblue, linestyle=:dash, lw=2 )
plot!( plt[2], [1,N], fill( λ2list[i], 2 ); color=:cornflowerblue, linestyle=:dash, lw=2 )
plot!( plt[3], [1,N], fill(  κlist[i], 2 ); color=:cornflowerblue, linestyle=:dash, lw=2 )

scatter!( plt[3], 1:N, θdata[i][:,3]; color=:gray )
plot!( plt[3]; xlims=(1,N), ylims=(0,1) )

In [ ]:
# Compile the approximated rate variables.
θ̂data = [[fill( λ1list[i], N ) fill( λ2list[i], N ) fill( κlist[i], N )] for i ∈ 1:Nρ];

In [ ]:
# Save parameter data.
if true
    # Save the cluster change parameters for each parameter case.
    for i ∈ 1:Nρ
        writedlm( folderlist[i]*"optimal-rates-hat.txt", θ̂data[i] )
    end
end